# 07 - Final segments: stability, outcomes and `customer_segments` (DS, Days 10-11)

The DA plan's Day 10-11 tasks for DS, done on the chosen candidate (**D, two-stage k-means**):

1. **Name the segments** from their profiles, with a check that the names still match the clusters.
2. **Stability (Day 10):** re-cluster 20 random 80% subsamples and check each segment survives. This is stronger than the random-start check in notebook 06.
3. **Do the segments differ on outcomes? (Day 10)** Conversion tested with a chi-square test and Cramer's V. This is the first time outcomes are used. They were kept out of the features on purpose.
4. **Segment statistics (Day 11):** size, budget, region, channel, conversion, deal value, speed, and payment record per segment.
5. **Write `customer_segments`** with `model_version = seg-v1.0`.

**Rules:** the answer key (`_restricted/`) is still not opened. It is run once, at the very end, only when the model is final.

## 0. Setup

In [ ]:
%pip install -q scikit-learn scipy

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from scipy.stats import chi2_contingency
from pandas.api.types import is_numeric_dtype

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

ROOT = Path.cwd().parent
FEAT = ROOT / "data" / "features"
OUT = ROOT / "notebooks" / "outputs"
OUT.mkdir(exist_ok=True)
for f in ["labels_candidates.parquet", "customer_features_mixed.parquet", "customer_features.parquet", "customer_outcomes.parquet"]:
    assert (FEAT / f).exists(), f"{f} missing: run notebooks 04 to 06 first"

FINAL = "D"                 # CONFIG: the chosen candidate (a column of labels_candidates.parquet)
K, K_RENT = 8, 2            # must match notebook 06
K_BUY = K - K_RENT
MODEL_VERSION = "seg-v1.0"

X = pd.read_parquet(FEAT / "customer_features_mixed.parquet").set_index("customer_id")
F = pd.read_parquet(FEAT / "customer_features.parquet").set_index("customer_id").loc[X.index]
O = pd.read_parquet(FEAT / "customer_outcomes.parquet").set_index("customer_id").loc[X.index]
lab = pd.read_parquet(FEAT / "labels_candidates.parquet").set_index("customer_id")[FINAL].loc[X.index].astype(str)
print(f"{len(lab)} customers | final candidate {FINAL} | clusters: {sorted(lab.unique())}")

## 1. Rebuild the model matrix
The same corrected matrix as notebook 06 (no `has_*` flags, typical browsing values for customers with no sessions, categories weighted), needed for the stability re-clustering.

In [ ]:
cat_cols = [c for c in X.columns if not is_numeric_dtype(X[c])]
num_cols = [c for c in X.columns if c not in cat_cols]
no_sess, no_view = F["has_sessions"].eq(0), F["has_views"].eq(0)
Xn = X[num_cols].drop(columns=["has_sessions", "has_views"], errors="ignore").copy()
for c in ["n_sessions", "mean_pages", "mean_duration_min"]:
    if c in Xn:
        Xn.loc[no_sess, c] = Xn.loc[~no_sess, c].median()
if "n_views" in Xn:
    Xn.loc[no_view, "n_views"] = Xn.loc[~no_view, "n_views"].median()
M = pd.concat([Xn, pd.get_dummies(X[cat_cols], dtype=float) * 0.7071], axis=1)
intent = X["primary_intent"]
print("Model matrix:", M.shape)

## 2. Name the segments
The names come from the section 6 profiles in notebook 06. **They are my reading of the clusters, so treat them as provisional and agree them with the DA team** (the DA plan has them build persona cards on Day 12).
The check below makes the code stop if a name no longer fits its cluster, which would happen if the labels changed because notebook 06 was re-run on different data or settings.

In [ ]:
SEGMENT_NAMES = {            # CONFIG: provisional names, from the notebook 06 profiles
    "B0": "Downsizers",
    "B1": "Home movers",
    "B2": "Window shoppers",
    "B3": "First-time buyers",
    "B4": "Buy-to-let investors",
    "B5": "Overseas investors",
    "R0": "Corporate relocation lets",
    "R1": "Young renters",
}
VERIFY_NAMES = True

assert set(SEGMENT_NAMES) == set(lab.unique()), f"Names cover {sorted(SEGMENT_NAMES)} but the clusters are {sorted(lab.unique())}"

def quick_profile(l):
    p = pd.DataFrame(index=sorted(l.unique()))
    p["customers"] = l.value_counts().sort_index()
    p["share_%"] = (p["customers"] / len(l) * 100).round(1)
    p["buy_%"] = (F["primary_intent"].eq("Buy").groupby(l).mean() * 100).round(0)
    p["overseas_%"] = (F["is_overseas"].astype(float).groupby(l).mean() * 100).round(0)
    p["median_budget"] = F["budget_gbp"].groupby(l).median().round(0)
    p["budget_missing_%"] = (F["budget_missing"].groupby(l).mean() * 100).round(0)
    p["mobile_%"] = (F["share_mobile"].groupby(l).mean() * 100).round(0)
    p["new_build_%"] = (F["share_new_build"].groupby(l).mean() * 100).round(0)
    p["median_sessions"] = F["n_sessions"].groupby(l).median()
    return p

prof = quick_profile(lab)
prof.insert(0, "name", prof.index.map(SEGMENT_NAMES))
display(prof)

if VERIFY_NAMES:
    rules = {
        "R0": lambda p: p["buy_%"] < 5 and p["median_budget"] > 2000,
        "R1": lambda p: p["buy_%"] < 5 and p["median_budget"] < 2000,
        "B5": lambda p: p["overseas_%"] > 80,
        "B0": lambda p: p["buy_%"] > 95 and p["median_budget"] > 350000 and p["mobile_%"] < 35,
        "B4": lambda p: p["buy_%"] > 95 and p["median_budget"] < 250000 and p["mobile_%"] < 35,
        "B3": lambda p: p["mobile_%"] > 70 and p["median_budget"] < 300000,
        "B2": lambda p: p["budget_missing_%"] > 50,
        "B1": lambda p: p["median_budget"] > 420000,
    }
    bad = [k for k, r in rules.items() if not r(prof.loc[k])]
    assert not bad, f"These names no longer fit their clusters: {bad}. Re-assign SEGMENT_NAMES from the table above."
    print("All segment names match their profiles.")

## 3. Stability: do the segments survive subsampling? (Day 10)
Re-cluster 20 random 80% subsamples (same two-stage method) and compare with the final labels on the customers in each subsample.
- **Adjusted Rand index (ARI):** overall agreement. 1 = identical.
- **Jaccard, per segment:** how much of each segment reappears as one cluster. Rule of thumb: **above 0.75 stable, 0.6 to 0.75 moderate, below 0.6 unstable.**

In [ ]:
def two_stage_fit(idx, seed, n_init=3):
    out = pd.Series(index=idx, dtype=object)
    for it, k, tag in [("Rent", K_RENT, "R"), ("Buy", K_BUY, "B")]:
        part_idx = idx[intent.loc[idx].eq(it).values]
        out.loc[part_idx] = [f"{tag}{c}" for c in KMeans(n_clusters=k, n_init=n_init, random_state=seed).fit_predict(M.loc[part_idx])]
    return out

rng = np.random.default_rng(42)
B_RUNS = 20
aris, jac = [], {c: [] for c in sorted(lab.unique())}
for b in range(B_RUNS):
    sub = M.index[np.sort(rng.choice(len(M), size=int(0.8 * len(M)), replace=False))]
    lb = two_stage_fit(sub, seed=b)
    orig = lab.loc[sub]
    aris.append(adjusted_rand_score(orig, lb))
    for c in jac:
        in_c = (orig == c).values
        jac[c].append(max((in_c & (lb == bc).values).sum() / (in_c | (lb == bc).values).sum() for bc in lb.unique()))

stab = pd.DataFrame({"segment": pd.Series(SEGMENT_NAMES), "mean_jaccard": {c: np.mean(v) for c, v in jac.items()},
                     "min_jaccard": {c: np.min(v) for c, v in jac.items()}}).round(3)
stab["verdict"] = np.where(stab["mean_jaccard"] > 0.75, "stable", np.where(stab["mean_jaccard"] >= 0.6, "moderate", "unstable"))
print(f"Overall ARI vs final labels over {B_RUNS} subsamples: mean {np.mean(aris):.3f} (min {np.min(aris):.3f}, max {np.max(aris):.3f})")
stab

## 4. Do the segments differ on outcomes? (Day 10)
Customers who made at least one lead are the only ones who *could* convert, so they are the base for conversion.
A chi-square test asks whether conversion is the same in every segment. **Cramer's V** says how big the difference is (0 = none, about 0.1 small, 0.3 medium, 0.5 large). A tiny p-value alone proves little with 60,000 customers; look at V and at the gaps in the table.

In [ ]:
D = O.join(lab.rename("segment_id")).join(F[["budget_gbp"]])
lead = D[D["n_leads_total"] > 0]
ct = pd.crosstab(lead["segment_id"], lead["converted_any"])
chi2, p, dof, _ = chi2_contingency(ct)
cramers_v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
print(f"Customers with a lead: {len(lead)} | chi-square {chi2:,.0f}, dof {dof}, p = {p:.2g} | Cramer's V = {cramers_v:.3f}")
overall = lead["converted_any"].mean() * 100
by = lead.groupby("segment_id")["converted_any"].mean().mul(100).round(1)
print(f"Overall conversion (customers with a lead): {overall:.1f}%")
by.rename(index=SEGMENT_NAMES).sort_values(ascending=False).to_frame("conversion_%")

In [ ]:
order = by.sort_values().index
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh([SEGMENT_NAMES[i] for i in order], by.loc[order], color="#1f4e79")
ax.axvline(overall, color="#d62728", linestyle="--", label=f"overall {overall:.0f}%")
ax.set(title="Conversion by segment (customers with at least one lead)", xlabel="% who converted")
ax.legend(); plt.tight_layout(); plt.savefig(OUT / "segment_conversion.png", dpi=150); plt.show()

## 5. Segment statistics (Day 11)
One row per segment, in the units the DA plan lists: size, budget, region, channel, conversion, deal value, speed, and payment record.
Use **medians** for money, because a few typo or very high values distort means.

In [ ]:
def top_share(s, n=3):
    vc = s.value_counts(normalize=True).head(n)
    return ", ".join(f"{k} {v:.0%}" for k, v in vc.items())

won = D[D["n_won"] > 0]
stats = pd.DataFrame(index=sorted(lab.unique()))
stats["segment"] = pd.Series(SEGMENT_NAMES)
stats["customers"] = lab.value_counts().sort_index()
stats["share_%"] = (stats["customers"] / len(lab) * 100).round(1)
stats["median_budget_gbp"] = D["budget_gbp"].groupby(D["segment_id"]).median().round(0)
stats["top_regions"] = F["uk_region_of_residence"].fillna("Overseas").groupby(lab).agg(top_share)
stats["top_lead_sources"] = F["primary_lead_source"].fillna("No lead").groupby(lab).agg(top_share)
stats["with_a_lead_%"] = ((D["n_leads_total"] > 0).groupby(D["segment_id"]).mean() * 100).round(1)
stats["conversion_%"] = by
stats["win_rate_%"] = (D.groupby("segment_id")["n_won"].sum() / D.groupby("segment_id")["n_leads_total"].sum() * 100).round(1)
stats["median_days_to_decision"] = won.groupby("segment_id")["days_to_decision"].median().round(0)
stats["median_sale_price_gbp"] = D.groupby("segment_id")["agreed_price_total_gbp"].median().round(0)
stats["median_rent_pcm_gbp"] = D.groupby("segment_id")["agreed_rent_pcm_total_gbp"].median().round(0)
tot_sale, tot_rent = D["agreed_price_total_gbp"].sum(), D["agreed_rent_pcm_total_gbp"].sum()
stats["share_of_sale_value_%"] = (D.groupby("segment_id")["agreed_price_total_gbp"].sum() / tot_sale * 100).round(1)
stats["share_of_rent_roll_%"] = (D.groupby("segment_id")["agreed_rent_pcm_total_gbp"].sum() / tot_rent * 100).round(1)
pay_ok, pay_late = D.groupby("segment_id")["payments_on_time"].sum(), D.groupby("segment_id")["payments_late_or_overdue"].sum()
stats["late_or_overdue_payments_%"] = (pay_late / (pay_ok + pay_late) * 100).round(1)
stats.to_csv(OUT / "segment_statistics.csv")
stats.drop(columns=["top_regions", "top_lead_sources"])

In [ ]:
stats[["segment", "top_regions", "top_lead_sources"]]

## 6. Write `customer_segments`
The table the DA team loads into PostgreSQL (Day 10): one row per customer, tagged with the model version so later versions can sit beside it.

In [ ]:
seg = pd.DataFrame({
    "customer_id": lab.index,
    "segment_id": lab.values,
    "segment_name": lab.map(SEGMENT_NAMES).values,
    "model_version": MODEL_VERSION,
    "assigned_on": pd.Timestamp.today().date(),
})
seg.to_parquet(FEAT / "customer_segments.parquet", index=False)
seg.to_csv(OUT / f"customer_segments_{MODEL_VERSION}.csv", index=False)
print(seg.shape, "->", FEAT / "customer_segments.parquet")
seg["segment_name"].value_counts().to_frame("customers")

## 7. Validate

In [ ]:
results = {
    "every customer has exactly one segment": seg["customer_id"].is_unique and len(seg) == len(X) and seg["segment_name"].notna().all(),
    "model_version is set on every row": bool((seg["model_version"] == MODEL_VERSION).all()),
    "no segment under 1% of customers": bool((stats["share_%"] >= 1).all()),
    "every segment is at least moderately stable (Jaccard >= 0.6)": bool((stab["mean_jaccard"] >= 0.6).all()),
    "overall subsample stability is high (ARI >= 0.8)": bool(np.mean(aris) >= 0.8),
    "segments differ on conversion (p < 0.01)": bool(p < 0.01),
    "the difference is not trivial (Cramer's V >= 0.1)": bool(cramers_v >= 0.1),
    "answer key never read": True,
}
for k_, v in results.items():
    print(("PASS  " if v else "CHECK ") + k_)
print("\n'CHECK' is something to look at and write down, not an error.")

## Hand-off notes
- **To the DA team (Day 10):** `data/features/customer_segments.parquet` (or the CSV in `outputs/`) and `outputs/segment_statistics.csv`. Their persona cards (Day 12) will use the provisional names, so agree the names with them.
- **Write down:** why D was chosen (stability 0.94, buy/rent separated, agrees with k-prototypes at 0.86), the stability result from section 3, the conversion test result, and which names are provisional.
- **Final scoring:** only when the segmentation is final, run `recoverability_check.py` **once** against the answer key and report the ARI as the project result (target: 0.5 or above). Do not change the model after seeing it.
- Add `data/features/` and `notebooks/outputs/` to `.gitignore` before you commit.